# EXP02 — Sensibilidade ao filtro de transações

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## Baixar e conferir os dados

Os arquivos processados do EXP01 e EXP02 estão disponíveis para **leitura pública** no Google Drive da pesquisa (links em `docs/DADOS.md`). A célula seguinte tenta baixar os dois Parquets e compara seus hashes SHA-256 com os registros históricos. Se o Google Drive bloquear o download, no Colab você poderá enviar os arquivos manualmente.

Dados já presentes com hashes corretos são reutilizados. A verificação de integridade não valida a qualidade dos rótulos nem refaz o processamento bruto da blockchain.


In [ ]:
from pathlib import Path
import hashlib
import tempfile
import os
import gdown

# Arquivos processados, publicados para leitura pela equipe da pesquisa.
ARQUIVOS = {
    "features_treino.parquet": {
        "id": "1jUHAq6H9FKSgptv4nsCoOmFILH4Py_2S",
        "sha256": "0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32",
    },
    "features_teste_20201029.parquet": {
        "id": "1g22eCccxC4v_S2RmxPhFZyzfF1ARuzPk",
        "sha256": "6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51",
    },
}

pasta_dados = RAIZ / "data" / "processed"
pasta_dados.mkdir(parents=True, exist_ok=True)

def conferir_hash(caminho, esperado):
    resumo = hashlib.sha256()
    with caminho.open("rb") as entrada:
        for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
            resumo.update(bloco)
    if resumo.hexdigest() != esperado:
        raise ValueError(f"O arquivo {caminho.name} não corresponde ao SHA-256 esperado.")

def baixar_e_conferir(nome, info):
    destino = pasta_dados / nome
    if destino.is_file():
        conferir_hash(destino, info["sha256"])
        print(f"Arquivo local conferido: {nome}")
        return

    temporario = pasta_dados / (nome + ".download")
    try:
        print(f"Baixando {nome} do Google Drive público...")
        resultado = gdown.download(id=info["id"], output=str(temporario), quiet=False)
        if not resultado or not temporario.is_file():
            raise RuntimeError("O Drive não forneceu o arquivo.")
        conferir_hash(temporario, info["sha256"])
        temporario.replace(destino)
        print(f"Download conferido: {nome} ({destino.stat().st_size} bytes)")
    finally:
        temporario.unlink(missing_ok=True)

falhas = []
for nome, info in ARQUIVOS.items():
    try:
        baixar_e_conferir(nome, info)
    except Exception as erro:
        if (pasta_dados / nome).exists():
            raise  # Não ignorar arquivo local com hash incorreto.
        print(f"Não foi possível baixar {nome}: {erro}")
        falhas.append(nome)

if falhas:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Download indisponível. Baixe os arquivos indicados em docs/DADOS.md "
            "e coloque-os em data/processed/."
        ) from exc
    print("Selecione os arquivos que faltaram:", ", ".join(falhas))
    recebidos = files.upload()
    for nome in falhas:
        if nome not in recebidos:
            raise FileNotFoundError(f"Upload não incluiu {nome}")
        destino = pasta_dados / nome
        destino.write_bytes(recebidos[nome])
        try:
            conferir_hash(destino, ARQUIVOS[nome]["sha256"])
        except Exception:
            destino.unlink(missing_ok=True)
            raise
        print(f"Upload conferido: {nome}")

print("Todos os dados necessários foram conferidos.")


## 2. Dados e critério

Usamos o mesmo treino e o mesmo dia de 2020 do EXP01. Mudaremos **somente o mínimo de transações**, comparando 1, 5, 10 e 20. O limiar permanece `score > 0,50`.

In [ ]:
import pandas as pd
from ethmix.features import FEATURES
from ethmix.execucao import abrir_dados, treinar, prever

pasta = RAIZ / "data" / "processed"
treino = abrir_dados(pasta / "features_treino.parquet", ["label", "n_tx", *FEATURES])
teste = abrir_dados(pasta / "features_teste_20201029.parquet", ["recipient", "n_tx", "tornado", *FEATURES])
modelo, _ = treinar(treino, seed=42, min_tx=10)
registros = []
for minimo in [1, 5, 10, 20]:
    avaliados = prever(modelo, teste, min_tx=minimo, limiar=0.50)
    registros.append({"min_tx": minimo, "elegiveis": len(avaliados),
                      "alertas": int(avaliados["alerta"].sum())})
resultado = pd.DataFrame(registros)
display(resultado)

## 3. Visualizar

As contagens mostram o quanto o filtro altera a população avaliada. Menos alertas **não** significa, por si só, maior precisão.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(resultado["min_tx"].astype(str), resultado["alertas"])
ax.set_xlabel("Mínimo de transações")
ax.set_ylabel("Quantidade de alertas")
plt.show()
saida = RAIZ / "results" / "generated" / "EXP02"
saida.mkdir(parents=True, exist_ok=True)
resultado.to_csv(saida / "sensibilidade.csv", index=False)

## 4. Referência histórica

O relatório anterior registra **49, 14, 9 e 7 alertas** para mínimos de 1, 5, 10 e 20. A execução atual só confirma esses números se os arquivos, os parâmetros e o modelo forem equivalentes. O fato de o resultado de 1 transação permanecer abaixo de 63 mostra que o filtro, sozinho, não resolve a divergência.